In [1]:
from LogicaGiocoo_cobra2 import *
import unittest

Test Abilità e Statistiche

In [2]:
class TestPlayerAbilita(unittest.TestCase):

    def setUp(self):
        """Prepara un Player1 standard prima di ogni test."""
        self.player = Player1("Giada", 5)

    def test_abilita_max_cap(self):
        """Verifica che le abilità non superino mai il valore 10."""
        # Proviamo a settare un danno fuori scala
        self.player.danno = 100 
        # Il setter deve aver bloccato il valore a 10
        self.assertEqual(self.player.danno, 10, "Il danno non deve superare il cap di 10")

    def test_abilita_min_cap(self):
        """Verifica che le abilità non scendano mai sotto lo zero."""
        # Proviamo a settare un valore negativo
        self.player.intelligenza = -5
        # Il setter deve aver bloccato il valore a 0
        self.assertEqual(self.player.intelligenza, 0, "L'intelligenza non può essere negativa")

    def test_bonus_iniziale_mercenario(self):
        """Verifica l'impatto della scelta 'Mercenario' su moralità e danno."""
        assegna_moralita(self.player, "mercenario egoista")
        self.assertEqual(self.player.moralita, 2)
        self.assertEqual(self.player.danno, 2)

    def test_somma_danno_arma(self):
        """Verifica il calcolo matematico del danno totale: $Danno_{base} + Danno_{arma}$."""
        # 1. Prepariamo le statistiche
        self.player.danno = 5
        spada = SpadaBase() # Questa arma ha danno 6
        mostro = Goblin()   # Il Goblin parte con 40 HP
        
        # 2. Eseguiamo l'attacco
        danno_effettivo = self.player.attacca(mostro, spada)
        
        # 3. Verifichiamo i risultati
        self.assertEqual(danno_effettivo, 11) 
        self.assertEqual(mostro.hp, 29) 

if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

....
----------------------------------------------------------------------
Ran 4 tests in 0.002s

OK


Log: Statistiche assegnate per mercenario egoista: Danno 2
Giada attacca Goblin per 11 danni
Log: Goblin ha subito 11 danni. HP rimanenti: 29


Test Factory Mostri

In [3]:
class TestNPCExtra(unittest.TestCase):
    def setUp(self):
        """Prepara l'ambiente di test: player e NPC freschi per ogni esecuzione."""
        self.player = Player1("Giada", 5)
        self.saggio = VecchioSaggio()
        self.guardia = GuardiaCorrotta()

    def test_saggio_scelta_neutrale(self):
        """Verifica che la scelta opportunista aumenti furtività senza intaccare la moralità."""
        # Chiamiamo l'interazione con l'indice 2 (Cosa ottengo in cambio?)
        self.saggio.interagisci(self.player, 2)
        self.assertEqual(self.player.furtivita, 1)
        self.assertEqual(self.player.moralita, 5)

    def test_guardia_corruzione_stealth(self):
        """Verifica che la corruzione aumenti la furtività ma penalizzi l'etica."""
        # Chiamiamo l'interazione con l'indice 1 (Corruzione)
        self.guardia.interagisci(self.player, 1)
        self.assertEqual(self.player.furtivita, 2)
        self.assertEqual(self.player.moralita, 4)

    def test_scelta_aggressiva_guardia(self):
        """Verifica che lo scontro diretto premi il danno e l'onore."""
        # Indice 0: Ti sfido a duello!
        self.guardia.interagisci(self.player, 0)
        self.assertEqual(self.player.danno, 2)
        self.assertEqual(self.player.moralita, 6)

    def test_scelta_egoista_saggio(self):
        """Verifica che l'indifferenza verso il saggio riduca drasticamente la moralità."""
        # Indice 1: Non ho tempo per i mendicanti
        self.saggio.interagisci(self.player, 1)
        self.assertEqual(self.player.moralita, 3)
        self.assertEqual(self.player.danno, 1)

    def test_goblin_creation_stats(self):
        """Verifica che la Factory produca correttamente un Goblin con i suoi parametri base."""
        # Istanziamo il creatore specifico
        creator = GoblinCreator()
        # Chiamiamo il factory method per ottenere l'oggetto Mostro
        mostro = creator.factory_method()
        
        self.assertIsInstance(mostro, Goblin)
        self.assertEqual(mostro.hp, 40)
        self.assertEqual(mostro.danno, 10)

if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

.........
----------------------------------------------------------------------
Ran 9 tests in 0.005s

OK


Log: Giada ha scelto: (Prova a corromperlo con l'oro)
Log: Giada ha scelto: Cosa ottengo in cambio?
Log: Giada ha scelto: Ti sfido a duello!
Log: Giada ha scelto: Non ho tempo per i mendicanti.
Log: Statistiche assegnate per mercenario egoista: Danno 2
Giada attacca Goblin per 11 danni
Log: Goblin ha subito 11 danni. HP rimanenti: 29


Test Abstract Factory

In [4]:
class TestItemsAndProtection(unittest.TestCase):    
    def setUp(self):
        """
        Configurazione iniziale eseguita prima di ogni test.
        Inizializziamo un Player1 con moralità 5.
        """
        self.player = Player1("Giada", 5)

    def test_armatura_base_logic(self):
        """Verifica riduzione del 5% (ratio 0.95) fornita dall'ArmaturaBase"""
        armatura = ArmaturaBase()
        # Calcolo: int(100 * 0.95) = 95
        self.assertEqual(armatura.difendi(100), 95)

    def test_armatura_elevata_reduction(self):
        """Verifica che l'ArmaturaElevata riduca il danno all'8% (ratio 0.08)"""
        armatura = ArmaturaElevata()
        self.player.equip_armatura(armatura)
        
        # Simuliamo un colpo da 100 danni: deve restituire 8
        danno_subito = self.player.take_damage(100)
        self.assertEqual(danno_subito, 8, "L'armatura deve filtrare il colpo")
        self.assertEqual(self.player.hp, 92)

    def test_integrazione_player_danno(self):
        """Verifica l'integrazione tra Player e Armatura durante un attacco subìto"""
        self.player.equip_armatura(ArmaturaElevata()) # Ratio 0.08
        # Attacco da 50 danni: int(50 * 0.08) = 4
        danno_ricevuto = self.player.take_damage(50)
        self.assertEqual(danno_ricevuto, 4)
        self.assertEqual(self.player.hp, 96)

    def test_livello1_empty_slots(self):
        """Verifica che il Livello 1 non fornisca aiuti difensivi o cure"""
        factory = Livello1Item()
        self.assertIsNone(factory.create_pozione(), "Il livello 1 non deve avere pozioni")
        self.assertIsNone(factory.create_armatura(), "Il livello 1 non deve avere armature")

    def test_livello4_factory(self):
        """Verifica che la Factory di Livello 4 crei lo Sniper e l'Armatura Finale"""
        factory = Livello4Item()
        arma = factory.create_arma()
        armatura = factory.create_armatura()
        
        self.assertIsInstance(arma, HeavySniper)
        self.assertIsInstance(armatura, ArmaturaPiuElevata)

if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

..............
----------------------------------------------------------------------
Ran 14 tests in 0.009s

OK


Log: Giada ha un'armatura che riduce il danno.
Log: Giada ha subito 8 danni. HP rimanenti: 92
Log: Giada ha un'armatura che riduce il danno.
Log: Giada ha subito 4 danni. HP rimanenti: 96
Log: Giada ha scelto: (Prova a corromperlo con l'oro)
Log: Giada ha scelto: Cosa ottengo in cambio?
Log: Giada ha scelto: Ti sfido a duello!
Log: Giada ha scelto: Non ho tempo per i mendicanti.
Log: Statistiche assegnate per mercenario egoista: Danno 2
Giada attacca Goblin per 11 danni
Log: Goblin ha subito 11 danni. HP rimanenti: 29


Test NPC

In [5]:
class TestNPCInteraction(unittest.TestCase):
    def setUp(self):
        """Configurazione iniziale per ogni test: Player con moralità 5 e NPC puliti."""
        self.player = Player1("Giada", 5) 
        self.saggio = VecchioSaggio()
        self.guardia = GuardiaCorrotta()

    def test_effetto_dialogo_applicato(self):
        """Verifica che la scelta 0 del Saggio aumenti moralità (+2) e intelligenza (+1)."""
        # Eseguiamo l'interazione con l'indice 0: 'Condivido il pane'
        self.saggio.interagisci(self.player, 0)
        
        self.assertEqual(self.player.moralita, 7)
        self.assertEqual(self.player.intelligenza, 1)

    def test_scelta_aggressiva_guardia(self):
        """Verifica che sfidare la guardia (indice 0) aumenti danno (+2) e moralità (+1)."""
        # Eseguiamo l'interazione: 'Ti sfido a duello!'
        self.guardia.interagisci(self.player, 0)
        
        self.assertEqual(self.player.danno, 2)
        self.assertEqual(self.player.moralita, 6)

    def test_scelta_egoista_saggio(self):
        """Verifica che ignorare il saggio (indice 1) riduca moralità (-2) e aumenti danno (+1)."""
        # Eseguiamo l'interazione: 'Non ho tempo per i mendicanti'
        self.saggio.interagisci(self.player, 1)
        
        self.assertEqual(self.player.moralita, 3)
        self.assertEqual(self.player.danno, 1)

if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

.................
----------------------------------------------------------------------
Ran 17 tests in 0.012s

OK


Log: Giada ha un'armatura che riduce il danno.
Log: Giada ha subito 8 danni. HP rimanenti: 92
Log: Giada ha un'armatura che riduce il danno.
Log: Giada ha subito 4 danni. HP rimanenti: 96
Log: Giada ha scelto: (Prova a corromperlo con l'oro)
Log: Giada ha scelto: Cosa ottengo in cambio?
Log: Giada ha scelto: Ti sfido a duello!
Log: Giada ha scelto: Non ho tempo per i mendicanti.
Log: Giada ha scelto: Condivido il mio pane con te.
Log: Giada ha scelto: Ti sfido a duello!
Log: Giada ha scelto: Non ho tempo per i mendicanti.
Log: Statistiche assegnate per mercenario egoista: Danno 2
Giada attacca Goblin per 11 danni
Log: Goblin ha subito 11 danni. HP rimanenti: 29
